# Classification Metrics from Scratch

Accuracy hides class imbalance. This notebook builds the confusion matrix, precision, recall, F1 (binary and macro/micro/weighted), ROC curve + AUC and precision-recall curve in numpy, and checks each against scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics as skm

## Confusion matrix and per-class scores

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if n_classes is None:
        n_classes = int(max(y_true.max(), y_pred.max())) + 1
    cm = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)   # rows = actual, cols = predicted
    return cm

def _safe_div(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.divide(a, b, out=np.zeros_like(a), where=b != 0)

def precision_recall_f1(y_true, y_pred, average=None, n_classes=None):
    cm = confusion_matrix(y_true, y_pred, n_classes)
    tp = np.diag(cm).astype(float)
    pred_pos, actual_pos = cm.sum(axis=0), cm.sum(axis=1)
    if average == 'micro':
        p = r = tp.sum() / cm.sum()
        return p, r, _safe_div(2 * p * r, p + r)
    p, r = _safe_div(tp, pred_pos), _safe_div(tp, actual_pos)
    f = _safe_div(2 * p * r, p + r)
    if average is None:
        return p, r, f
    w = np.ones(len(tp)) / len(tp) if average == 'macro' else actual_pos / actual_pos.sum()
    return (p * w).sum(), (r * w).sum(), (f * w).sum()

def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

## ROC and precision-recall curves

Sort samples by score descending; each distinct score is a threshold, and cumulative TP/FP counts give every curve point in one pass.

In [ ]:
def _threshold_counts(y_true, scores):
    y_true, scores = np.asarray(y_true), np.asarray(scores)
    order = np.argsort(-scores, kind='mergesort')
    y_s, s_s = y_true[order], scores[order]
    distinct = np.where(np.diff(s_s))[0]
    idx = np.r_[distinct, len(y_s) - 1]       # last index of each distinct score
    tps = np.cumsum(y_s)[idx]
    fps = 1 + idx - tps
    return fps, tps, s_s[idx]

def roc_curve(y_true, scores):
    fps, tps, thr = _threshold_counts(y_true, scores)
    tpr, fpr = tps / tps[-1], fps / fps[-1]
    return np.r_[0, fpr], np.r_[0, tpr], np.r_[np.inf, thr]

def auc(x, y):
    return float(np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2))   # trapezoid rule

def roc_auc_score(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return auc(fpr, tpr)

def precision_recall_curve(y_true, scores):
    fps, tps, thr = _threshold_counts(y_true, scores)
    precision, recall = tps / (tps + fps), tps / tps[-1]
    return np.r_[precision[::-1], 1], np.r_[recall[::-1], 0], thr[::-1]

def average_precision(y_true, scores):
    p, r, _ = precision_recall_curve(y_true, scores)
    return float(-np.sum(np.diff(r) * p[:-1]))

## Train a scratch logistic regression for scores

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
sc = StandardScaler().fit(X_tr)
X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

w, b = np.zeros(X_tr.shape[1]), 0.0
for _ in range(1000):
    err = sigmoid(X_tr @ w + b) - y_tr
    w -= 0.1 * X_tr.T @ err / len(y_tr)
    b -= 0.1 * err.mean()

scores = sigmoid(X_te @ w + b)
pred = (scores >= 0.5).astype(int)
print('accuracy', accuracy(y_te, pred))
print(confusion_matrix(y_te, pred))

## Verify against scikit-learn

In [ ]:
p, r, f = precision_recall_f1(y_te, pred)
sp, sr, sf, _ = skm.precision_recall_fscore_support(y_te, pred)
assert np.allclose(p, sp) and np.allclose(r, sr) and np.allclose(f, sf)
for avg in ['micro', 'macro', 'weighted']:
    mine = precision_recall_f1(y_te, pred, average=avg)
    ref = skm.precision_recall_fscore_support(y_te, pred, average=avg)[:3]
    assert np.allclose(mine, ref), avg
assert np.array_equal(confusion_matrix(y_te, pred), skm.confusion_matrix(y_te, pred))
assert np.isclose(roc_auc_score(y_te, scores), skm.roc_auc_score(y_te, scores))
assert np.isclose(average_precision(y_te, scores), skm.average_precision_score(y_te, scores))
print('ROC AUC', roc_auc_score(y_te, scores), '| AP', average_precision(y_te, scores))
print('all metrics match scikit-learn')

## Curves

On an imbalanced problem accuracy looks great while recall collapses; the threshold sweep shows the trade-off.

In [ ]:
fpr, tpr, _ = roc_curve(y_te, scores)
pr, rc, _ = precision_recall_curve(y_te, scores)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.3f}'); ax[0].plot([0, 1], [0, 1], 'k--')
ax[0].set(xlabel='False positive rate', ylabel='True positive rate', title='ROC'); ax[0].legend()
ax[1].plot(rc, pr); ax[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall')
plt.tight_layout(); plt.show()

In [ ]:
# Imbalance demo: always predicting the majority class
rng = np.random.RandomState(0)
y_imb = (rng.rand(1000) < 0.05).astype(int)
dumb = np.zeros_like(y_imb)
print('accuracy', accuracy(y_imb, dumb))
print('precision/recall/f1 of positive class', [v[1] for v in precision_recall_f1(y_imb, dumb, n_classes=2)])